# Train a Regression Model

In this notebook we will train a Linear Regression model on the Green Taxi dataset. We will only use one month for the training and keep only a small number of features. 

We want the model to **predict the duration of a trip**. This can be useful for the taxi drivers to plan their trips, for the customers to know how long a trip will take but also for the taxi companies to plan their fleet. The first two predictions would need real time predictions because the duration of a trip is not known in advance. The last one could be done in batch mode, as it is more a analytical task that doesn't need to be done in real time.

Additionally, we will use MLflow to track the model training and log the model artifacts.

In [ ]:
import os
from dotenv import load_dotenv

import pandas as pd

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline,make_pipeline

In [ ]:
year = 2025
month = 7
color = "green"

The variables `color`, `year` and `month` are used to specify the dataset we want to use to train the model.

In [ ]:
# Download the data
if not os.path.exists(f"./data/{color}_tripdata_{year}-{month:02d}.parquet"):
    os.system(f"wget -P ./data https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
# Load the data
df = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
df.shape

Now we will set up the connection to MLflow. For that we have to create a `.env` file with the URI to the MLflow Server in GCP (this will be `http://<external-ip>:5000`). You can simply run:

```bash
echo "MLFLOW_TRACKING_URI=http://<external-ip>:5000" > .env
```

We also will create an experiment to track the model and the metrics.

In [ ]:
load_dotenv()

MLFLOW_TRACKING_URI=os.getenv("MLFLOW_TRACKING_URI")

In the next cell we will **set up the connection to MLflow** and **create an experiment**. In order to create an experiment you need to provide:
- `EXPERIMENT_NAME`: the name of the experiment. You can choose any name you want. If the experiment already exists, it will be used.
- `MLFLOW_TRACKING_URI`: the URI of the MLflow server. This is the one you set in the `.env` file.
- `artifact_location`: the location where the artifacts will be stored. This is the GCP bucket you created. Make sure to replace `your-bucket-name` with the name of your bucket. If you created a folder inside the bucket to store the artifacts, you can add it to the path (e.g. `gs://your-bucket-name/models`). 

Remember that when we started the MLflow server on the GCP engine we set the `--default-artifact-root` to the same bucket. 

In [ ]:
# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Give the experiment a name
experiment_name = f"{color}-taxi-experiment-1"
# Get the experiment by name
exp = mlflow.get_experiment_by_name(experiment_name)

# Create the experiment if it does not exist
if exp is None:
    mlflow.create_experiment(name=experiment_name, 
                         tags={"developer": "Carmine Somma"}, 
                         artifact_location="gs://mlflows-artifacts/models"
                         )
else:
    experiment_id = exp.experiment_id
    print(f"Using existing experiment: {experiment_name} (ID: {experiment_id})")

# Set the experiment for logging
mlflow.set_experiment(experiment_name)

If everything went well, you should be able to see the experiment now in the MLflow UI at `http://<external-ip>:5000`.

Let's start now with looking at the data a bit:

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Look for missing values
df.isnull().sum()

Nearly all features seem to be in the correct type and we have only missings in features that we will not use for the model training. For predicting the duration of a trip, we will use the following features:

- `PULocationID`: The pickup location ID
- `DOLocationID`: The dropoff location ID
- `trip_distance`: The distance of the trip in miles

But first we have to calculate the duration of the trip in minutes because it is our target. For that we will use the `tpep_pickup_datetime` and `tpep_dropoff_datetime` columns. We will also remove all trips that have a duration of 0 and that are longer than 1 hours to remove outliers.

Additionally we will transform `DOLocationID` and `PULocationID` to categorical features. And combine them to a new feature `trip_route` that will contain the route of the trip.

In [ ]:
features = ["PULocationID", "DOLocationID", "trip_distance"]
target = 'duration'

In [ ]:
# Calculate the trip duration in minutes and drop trips that are less than 1 minute and more than 2 hours
def calculate_trip_duration_in_minutes(df):
    df["trip_duration_minutes"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60
    df = df[(df["trip_duration_minutes"] >= 1) & (df["trip_duration_minutes"] <= 60)]
    return df

In [ ]:
def preprocess(df):
    df = df.copy()
    df = calculate_trip_duration_in_minutes(df)
    categorical_features = ["PULocationID", "DOLocationID"]
    df[categorical_features] = df[categorical_features].astype(str)
    df['trip_route'] = df["PULocationID"] + "_" + df["DOLocationID"]
    df = df[['trip_route', 'trip_distance', 'trip_duration_minutes']]
    return df

In [ ]:
df_processed = preprocess(df)

Now that we have the dataframe that we want to train our model on, we need to split it into a train and test set. We will use 80% of the data for training and 20% for testing.

In [ ]:
y = df_processed["trip_duration_minutes"]
X = df_processed.drop(columns=["trip_duration_minutes"])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

We will now combine the `trip_distance` and the `trip_route` in a dictionary and transform it with the [DictVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.DictVectorizer.html) from `sklearn` to a sparse matrix, which is basically a one-hot encoding of the categorical features and includes the distance.

In [ ]:
dv = DictVectorizer()

dv.fit(X_train.to_dict(orient="records"))
X_train = dv.transform(X_train.to_dict(orient="records"))
X_test = dv.transform(X_test.to_dict(orient="records"))

And now we can **train the model** and **track the experiment with MLflow**. We will set tags to the experiment to make it easier to find it later.

- `model`: `linear-regression`
- `dataset`: `green-taxi`
- `developer`: `your-name`
- `train_size`: The size of the train set
- `test_size`: The size of the test set
- `features`: The features that we used for training
- `target`: The target that we want to predict
- `year`: The year of the data
- `month`: The month of the data

We could also log the model parameters but Linear Regression doesn't have any.

And finally we will log the metrics:

- `rmse`: The root mean squared error

We will also log the model artifacts. For that we will need to set the `service account json` that we downloaded earlier as the environment variable `GOOGLE_APPLICATION_CREDENTIALS`. 

Add in the `.env` file:

`GOOGLE_APPLICATION_CREDENTIALS=path/to/your/service-account-file.json
`

In case you have stored the file in the `sa_key/` folder in the repo, you can add in the `.env` file:

`GOOGLE_APPLICATION_CREDENTIALS=sa_key/your-service-account-file.json
`

In [ ]:
GOOGLE_APPLICATION_CREDENTIALS = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

Give Storage permission to the Compute Engine in order for the artifacts generated after the MLflow run gets storaged in the Cloud Storage bucket.

![storage_permission](./images/storage_permission.png)

In [ ]:
# If the connection and permissions are correct you should be able to run the below code without any errors
import tempfile
with mlflow.start_run():
    with tempfile.NamedTemporaryFile("w", delete=False) as f:
        f.write("test artifact")
        mlflow.log_artifact(f.name)

Next, we will train the model and log the experiment to MLflow.  

To log the model we will use `mlflow.sklearn.log_model` and we will give the model a name `mlflow-model-v1`. You can choose any name you want. This name will be used to identify the model in the MLflow UI as well as to load the model later for predictions (as we do in the notebook [03-batch-deployment.ipynb](./03-batch-deployment.ipynb)).  

If you log a model with the same name again, it will create a new version of the model. Each version of the model will have a unique `RUN_ID` that you can use to load the model later for predictions. You can find the `RUN_ID` in the MLflow UI, in the "Experiments" section. It is a string of 32 characters.

In [ ]:
with mlflow.start_run():
    
    tags = {
        "model": "linear regression",
        "developer": "<your name>",
        "dataset": f"{color}-taxi",
        "year": year,
        "month": month,
        "features": features,
        "target": target
    }
    mlflow.set_tags(tags)
    
    lr = LinearRegression()
    sk_model = lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)   
    
    mlflow.sklearn.log_model(sk_model=sk_model, name="mlflow-model-v1")

You should now see your run in the MLflow UI. Under the created experiment, you can also see the logged tags, the metric and the saved model.

![mlflow-ui](./images/mlflow-run.png)

And you can see what you need to do to load the model in an API or script in the UI as long as the application has access to MLflow.

But now let's add the `DictVectorizer` and the model to a pipeline and run the training again. First we need to create a new pair of train and test set because we will do the transformation in the pipeline:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [ ]:
X_train = X_train.to_dict(orient="records")
X_test = X_test.to_dict(orient="records")

In [ ]:
with mlflow.start_run():
    
    tags = {
        "model": "linear regression pipeline",
        "developer": "<your name>",
        "dataset": f"{color}-taxi",
        "year": year,
        "month": month,
        "features": features,
        "target": target
    }
    mlflow.set_tags(tags)
    
    pipeline = make_pipeline(
         DictVectorizer(),
        LinearRegression()
    )
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)

    sk_model_pipeline = pipeline
    
    mlflow.sklearn.log_model(sk_model=sk_model_pipeline, name="mlflow-model-v1")

Now you should see a new experiment with a new run id in MLflow. You can also see the pipeline and the model in the UI under `Artifacts`.

You may have noticed that the metric `rmse` is the same as before. This is because we have the same data,  the same model and the same parameters, the only difference is that we have added the transformation together with the model into a pipeline.